Imlo coursework

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [17]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cpu


In [18]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [19]:
#defining the train and val
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = transform,
    download = True
)

#splitting trainval
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_data, val_data = torch.utils.data.random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=32, shuffle=True, num_workers=2)

In [20]:
image, label = train_data[0]

In [21]:
image.size()

torch.Size([3, 128, 128])

In [22]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [23]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 5)

        self.fc1 = nn.Linear(24 * 29 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 37)

    def forward(self, input):
        input = self.pool(F.relu(self.conv1(input)))  #applying conv1, then RELU, then pooling layer
        input = self.pool(F.relu(self.conv2(input)))  #applying conv2, then RELU then pooling layer
        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        input = self.fc3(input)  #applying fc3
        return input

In [24]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.001)

In [26]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    running_loss_calc = running_loss / len(train_loader)
    print("Loss:", running_loss_calc)

Training epoch: 0
Loss: 3.5401029716367307
Training epoch: 1
Loss: 3.3554762653682544
Training epoch: 2
Loss: 3.0379614700441775
Training epoch: 3
Loss: 2.453141282434049
Training epoch: 4
Loss: 1.6740116243777068
Training epoch: 5
Loss: 0.9187514467731767
Training epoch: 6
Loss: 0.42638428501136927
Training epoch: 7
Loss: 0.1677839097202472
Training epoch: 8
Loss: 0.07570477628715984
Training epoch: 9
Loss: 0.07133360589226789
Training epoch: 10
Loss: 0.11280381475048869
Training epoch: 11
Loss: 0.1235825442607798
Training epoch: 12
Loss: 0.1369397325400749
Training epoch: 13
Loss: 0.1005349975362744
Training epoch: 14
Loss: 0.04498107773859216
Training epoch: 15
Loss: 0.0128298505442217
Training epoch: 16
Loss: 0.005510411673521562
Training epoch: 17
Loss: 0.0013103494320998636
Training epoch: 18
Loss: 0.0005645164894066629
Training epoch: 19
Loss: 0.00042207812570584633
Training epoch: 20
Loss: 0.00034084956765265974
Training epoch: 21
Loss: 0.0002855360313412308
Training epoch: 22


In [31]:
# testing the model on val data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in val_loader:
    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 11.277173913043478
